In [1]:
# ============================================================
# Stroke Risk Prediction (Improved Version)
# ============================================================

import pandas as pd
import numpy as np

# Preprocessing & ML
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif

# Classifiers
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.neural_network import MLPClassifier

# SMOTE for balancing
from imblearn.over_sampling import SMOTE

# ============================================================
# Load and Preprocess Dataset
# ============================================================
print("Loading dataset...")
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

# Drop rows with missing BMI and keep only adults
df = df.dropna(subset=["bmi"])
df = df[df["age"] > 18]

# Encode categorical variables
categorical_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# Separate features and target
X = df.drop(columns=["id", "stroke"])
y = df["stroke"]

# Balance dataset using SMOTE
print("Applying SMOTE balancing...")
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# ============================================================
# Feature Importance
# ============================================================
print("\nCalculating feature importance...")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_res, y_res)

rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "RandomForest Importance": rf.feature_importances_
}).sort_values(by="RandomForest Importance", ascending=False)

info_gain = mutual_info_classif(X_res, y_res, random_state=42)
ig_importance = pd.DataFrame({
    "Feature": X.columns,
    "Information Gain": info_gain
}).sort_values(by="Information Gain", ascending=False)

feature_importance = pd.merge(rf_importance, ig_importance, on="Feature")
print("\nFeature Importance (Random Forest vs Information Gain):\n")
print(feature_importance)



# ============================================================
# Define Machine Learning Models
# ============================================================
print("\nDefining models...")

scaler = StandardScaler()

models = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": Pipeline([
        ("scaler", scaler),
        ("clf", LogisticRegression(max_iter=1000))
    ]),
    "KNN": Pipeline([
        ("scaler", scaler),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),
    "SGD": Pipeline([
        ("scaler", scaler),
        ("clf", CalibratedClassifierCV(
            SGDClassifier(loss="log_loss",
                          penalty="elasticnet",alpha=0.0001,learning_rate="optimal",max_iter=2000,
                          tol=1e-4,random_state=42)
        ))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "MLP": Pipeline([
        ("scaler", scaler),
        ("clf", MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                              activation='relu', solver='adam',alpha=0.0005,learning_rate='adaptive',
                              learning_rate_init=0.001,max_iter=1500,early_stopping=True,random_state=42))
    ])
}

# Ensemble models
voting_clf = VotingClassifier(
    estimators=[
        ("nb", GaussianNB()),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
        ("dt", DecisionTreeClassifier(random_state=42))
    ],
    voting="soft"
)

stacking_clf = StackingClassifier(
    estimators=[
        ("nb", GaussianNB()),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=42)),
        ("dt", DecisionTreeClassifier(random_state=42))
    ],
    final_estimator=LogisticRegression(max_iter=1000)
)

models["Voting"] = voting_clf
models["Stacking"] = stacking_clf

# ============================================================
# 10-Fold Cross-Validation with AUC + Best Fold Tracking
# ============================================================
print("\nRunning 10-Fold Cross-Validation...")

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    print(f"\nEvaluating model: {name}")
    accs, precs, recs, f1s, aucs = [], [], [], [], []

    best_fold = None
    best_acc = 0.0

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_res, y_res), start=1):
        X_train, X_test = X_res.iloc[train_idx], X_res.iloc[test_idx]
        y_train, y_test = y_res.iloc[train_idx], y_res.iloc[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)[:, 1]
        elif hasattr(model, "decision_function"):
            y_prob = model.decision_function(X_test)
        else:
            y_prob = None

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob) if y_prob is not None else np.nan

        accs.append(acc)
        precs.append(prec)
        recs.append(rec)
        f1s.append(f1)
        aucs.append(auc)

        # Track best fold
        if acc > best_acc:
            best_acc = acc
            best_fold = fold_idx

    print(f" → Best Fold: {best_fold} ")

    results.append([
        name,
        round(np.mean(accs), 3),
        round(np.mean(precs), 3),
        round(np.mean(recs), 3),
        round(np.mean(f1s), 3),
        round(np.nanmean(aucs), 3),


    ])

results_df = pd.DataFrame(
    results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]
)

print("\n Machine Learning Model Performance (10-Fold Cross-Validation):\n")
print(results_df.sort_values(by="AUC", ascending=False))


Loading dataset...
Applying SMOTE balancing...

Calculating feature importance...

Feature Importance (Random Forest vs Information Gain):

             Feature  RandomForest Importance  Information Gain
0                age                 0.394367          0.503901
1  avg_glucose_level                 0.219878          0.066162
2                bmi                 0.147235          0.426562
3          work_type                 0.061003          0.054650
4     smoking_status                 0.059456          0.040435
5     Residence_type                 0.039047          0.035891
6             gender                 0.028049          0.019438
7       ever_married                 0.023263          0.013891
8       hypertension                 0.015247          0.004586
9      heart_disease                 0.012456          0.000000

Defining models...

Running 10-Fold Cross-Validation...

Evaluating model: Naive Bayes
 → Best Fold: 5 

Evaluating model: Logistic Regression
 → Best Fold